## Crop type mapping workflow 

### General
This Notebook runs the crop type classification workflow documented in from Akinyemi, Rufin, et al. 2026

### Requirements
Field samples for classification are available via [Zenodo repository](zenodo.org). This Notebook requires a Google Earth Engine (GEE) account with sufficient processing credits and a Google Cloud Project ID for initializing the ee Python module. Helper functions for feature computation and processing in GEE come from eepypr, further dependencies are listed below.

### Notes
Parallel computation of intermediate products (STM) and export as assets for improved efficiency. Cells ending with task.start() trigger computations in GEE, for monitoring runtime and success please visit GEE Task Manager. Final map can be retrieved from cloud storage after computation is complete.

### Import & initialize

In [2]:
import os
import sys

sys.path.append(os.path.abspath('../../'))

from src import sen, sar, stm, vec

In [ ]:
import ee
ee.Initialize(project='projectname')

import time
import datetime

import json
import geopandas as gpd
import pandas as pd
import numpy as np

### ROI

In [ ]:
# define roi boundaries as rectangle
roi = ee.Geometry.Rectangle([[3.2, 8.0],[3.4, 8.2]])

### S1 Features

In [ ]:

startDate = datetime.datetime(2021, 10, 1)
endDate = datetime.datetime(2023, 3, 31)

# where to store assets
asset_folder = 'users/username/s1_ard/' # must exist

tempbins = [[datetime.datetime(2021, 10, 1), datetime.datetime(2021, 10, 31), 's1_ard_2021-10'], 
            [datetime.datetime(2021, 11, 1), datetime.datetime(2021, 11, 30), 's1_ard_2021-11'],
            [datetime.datetime(2021, 12, 1), datetime.datetime(2021, 12, 31), 's1_ard_2021-12'],
            [datetime.datetime(2022,  1, 1), datetime.datetime(2022,  1, 31), 's1_ard_2022-01'],
            [datetime.datetime(2022,  2, 1), datetime.datetime(2022,  2, 28), 's1_ard_2022-02'],
            [datetime.datetime(2022,  3, 1), datetime.datetime(2022,  3, 31), 's1_ard_2022-03'],
            [datetime.datetime(2022,  4, 1), datetime.datetime(2022,  4, 30), 's1_ard_2022-04'],
            [datetime.datetime(2022,  5, 1), datetime.datetime(2022,  5, 31), 's1_ard_2022-05'],
            [datetime.datetime(2022,  6, 1), datetime.datetime(2022,  6, 30), 's1_ard_2022-06'],
            [datetime.datetime(2022,  7, 1), datetime.datetime(2022,  7, 31), 's1_ard_2022-07'],
            [datetime.datetime(2022,  8, 1), datetime.datetime(2022,  8, 31), 's1_ard_2022-08'],
            [datetime.datetime(2022,  9, 1), datetime.datetime(2022,  9, 30), 's1_ard_2022-09'],
            [datetime.datetime(2022, 10, 1), datetime.datetime(2022, 10, 31), 's1_ard_2022-10'],
            [datetime.datetime(2022, 11, 1), datetime.datetime(2022, 11, 30), 's1_ard_2022-11'],
            [datetime.datetime(2022, 12, 1), datetime.datetime(2022, 12, 31), 's1_ard_2022-12'],
            [datetime.datetime(2023,  1, 1), datetime.datetime(2023,  1, 31), 's1_ard_2023-01'],
            [datetime.datetime(2023,  2, 1), datetime.datetime(2023,  2, 28), 's1_ard_2023-02'],
            [datetime.datetime(2023,  3, 1), datetime.datetime(2023,  3, 31), 's1_ard_2023-03']]

In [ ]:
# config for s1 collection
collection = src.sar.s1_grd_asc(startDate, endDate)

# process features for temp bins
for s, [startDate, endDate, ref_img] in enumerate(tempbins, start=1):
    print('\nCreating S1 reference for bin ' + str(s))
    print('Starting ' + str(startDate) + ', ending ' + str(endDate))

    # filter collection
    ref = collection.filterDate(startDate, endDate).select(['VHg0', 'VVg0', 'Ratio'])
    # reduce mean
    img = ref.reduce(ee.Reducer.mean()).rename(['vh_avg', 'vv_avg', 'cr_avg'])

    # export as asset
    print('Storing reference as ' + ref_img)
    task = ee.batch.Export.image.toAsset(**{
        'image': img,
        'scale': 10,
        'region': roi,
        'description': ref_img,
        'assetId': asset_folder + ref_img,
        'maxPixels': 1e13
    })
    
    task.start()

### S2 features

In [ ]:
# define region of interest
roi = ee.Geometry.Rectangle([[3.2, 8.0],[3.4, 8.2]])

# where to store assets
asset_folder = 'users/username/s2_ard/' # must exist

In [ ]:
# define time range for temporal bins and corresponding asset names
tempbins = [[datetime.datetime(2021, 10, 1), datetime.datetime(2021, 11, 30), 's2_ard_2021-10-11'],
            [datetime.datetime(2021, 12, 1), datetime.datetime(2022,  1, 31), 's2_ard_2021-12-01'],
            [datetime.datetime(2022,  2, 1), datetime.datetime(2022,  3, 31), 's2_ard_2022-02-03'],
            [datetime.datetime(2022, 10, 1), datetime.datetime(2022, 11, 30), 's2_ard_2022-10-11'],
            [datetime.datetime(2022, 12, 1), datetime.datetime(2023,  1, 31), 's2_ard_2022-12-01'],
            [datetime.datetime(2023,  2, 1), datetime.datetime(2023,  3, 31), 's2_ard_2023-02-03']]

In [ ]:
# process features for temp bins
for s, [startDate, endDate, ref_img] in enumerate(tempbins, start=1):
    print('\nCreating S2 reference for bin ' + str(s))
    print('Starting ' + str(startDate) + ', ending ' + str(endDate))

    img = src.stm.SEN_STM(startDate, endDate,
                          bands=['blue', 'green', 'red', 'rededge1', 'rededge2', 'rededge3', 'nir', 'swir1', 'swir2', 'ndvi', 'ndmi'],
                          metrics=['median', 'sd', 'p25', 'p75', 'iqr', 'imean']).toInt16()

    # export as asset
    print('Storing reference as ' + ref_img)
    task = ee.batch.Export.image.toAsset(**{
        'image': img,
        'scale': 10,
        'region': roi,
        'description': ref_img,
        'assetId': asset_folder + ref_img,
        'maxPixels': 1e13
    })
    
    task.start()

### Crop type classification

In [ ]:
# local shapefile with training points
point_shape = '/training.gpkg'

# class name column
target = 'class_id'

# read training vector
points = gpd.read_file(point_shape)

In [ ]:
# crs manipulation
points.crs
if False:
    points = points.to_crs(epsg=4326)

In [ ]:
# print overview of class occurrence
pd.unique(points[target])
points[target].value_counts()
points[target].value_counts()

for col in points.columns:
    print(col)
points.head()

# remove other attributes
points = points.drop(['id', 'path'], axis=1)
points = points[points[target].notna()]


In [ ]:
# create geojson from geopandas
point_f = []
for i in range(points.shape[0]):
    geom = points.iloc[i:i + 1, :]
    jsonDict = eval(geom.to_json())
    geojsonDict = jsonDict['features'][0]
    point_f.append(ee.Feature(geojsonDict))

# make feature collection
ptsfc = ee.FeatureCollection(point_f)

In [ ]:
# stack s1 and s2 assets

feat = ['users/philipperufin/s2_ard/s2_ard_2021-10-11',
        'users/philipperufin/s2_ard/s2_ard_2021-12-01',
        'users/philipperufin/s2_ard/s2_ard_2022-02-03',
        'users/philipperufin/s2_ard/s2_ard_2022-10-11',
        'users/philipperufin/s2_ard/s2_ard_2022-12-01',
        'users/philipperufin/s2_ard/s2_ard_2023-02-03',

        'users/philipperufin/s1_ard/s1_ard_2021-10',
        'users/philipperufin/s1_ard/s1_ard_2021-11',
        'users/philipperufin/s1_ard/s1_ard_2021-12',
        'users/philipperufin/s1_ard/s1_ard_2022-01',
        'users/philipperufin/s1_ard/s1_ard_2022-02',
        'users/philipperufin/s1_ard/s1_ard_2022-03',
        'users/philipperufin/s1_ard/s1_ard_2022-04',
        'users/philipperufin/s1_ard/s1_ard_2022-05',
        'users/philipperufin/s1_ard/s1_ard_2022-06',
        'users/philipperufin/s1_ard/s1_ard_2022-07',
        'users/philipperufin/s1_ard/s1_ard_2022-08',
        'users/philipperufin/s1_ard/s1_ard_2022-09',
        'users/philipperufin/s1_ard/s1_ard_2022-10',
        'users/philipperufin/s1_ard/s1_ard_2022-11',
        'users/philipperufin/s1_ard/s1_ard_2022-12',
        'users/philipperufin/s1_ard/s1_ard_2023-01',
        'users/philipperufin/s1_ard/s1_ard_2023-02',
        'users/philipperufin/s1_ard/s1_ard_2023-03']

stm_image = ee.Image([feat])
bands = stm_image.bandNames().getInfo()
len(bands)


In [ ]:
# sample point locations
stm = stm_image.sampleRegions(ptsfc, [target], 10)

In [ ]:
# classifier
# define band names
bands = stm_image.bandNames().getInfo()
len(bands)

classifier = ee.Classifier.smileRandomForest(250).train(stm, target, bands, subsamplingSeed=42042)
multiprob = ee.Classifier.smileRandomForest(250).setOutputMode('MULTIPROBABILITY').train(stm, target, bands, subsamplingSeed=42042)


In [ ]:
oob = classifier.confusionMatrix().accuracy().getInfo()
con = classifier.confusionMatrix().getInfo()
var = classifier.explain().getInfo()

In [ ]:
# define roi boundaries
roi_path = '/roi.gpkg'
roi_vect = gpd.read_file(roi_path)

g = json.loads(roi_vect.to_json())
coords = list(g['features'][0]['geometry']['coordinates'])
roi_poly = ee.Geometry.Polygon(coords)

In [ ]:
### create classification
map = stm_image.classify(classifier).toInt8()

In [ ]:
# export as asset
task = ee.batch.Export.image.toAsset(**{
    'image': map,
    'scale': 10,
    'region': roi_poly,
    'description': 'crops',
    'assetId': 'crops',
    'maxPixels': 1e13
})

task.start()

In [ ]:
# create multiprobability outputs
prb = stm_image.classify(multiprob).arrayFlatten(coordinateLabels=[['cassava-ep', 'cassava-lp', 'maize-cassava', 'maize-ep', 'maize-lp', 'rice', 'yam', 'others']]).multiply(10000).toInt16()

# export as asset
task = ee.batch.Export.image.toAsset(**{
    'image': prb,
    'scale': 10,
    'region': roi_poly,
    'description': 'crops_probs',
    'assetId': 'crops_probs',
    'maxPixels': 1e13
})

task.start()